# Vision-Language Models — CLIP from scratch in NumPy

**Course:** [Computer Vision · Vision-Language Models](https://ml-viz-ruby.vercel.app/courses/computer-vision/04-vision-language-models)

**The idea in one sentence.** Train an image encoder and a text encoder so that
*matched* image–text pairs land close together in one shared vector space and
*mismatched* pairs land far apart. Once that space exists you get **zero-shot**
classification for free: to label an image, embed the candidate class *names* as
text and pick the nearest one — no classifier head, no fine-tuning.

**Why contrastive and not a classifier?** A softmax classifier can only name the
classes it was trained on. CLIP never learns a fixed label set — it learns a
*similarity function* between pixels and words, so at test time you can invent
new classes just by writing new prompts.

Roadmap for this notebook:

1. **From scratch** — two linear encoders into a shared $d$-dim space, the
   symmetric contrastive loss, and finite-difference training.
2. **Validate** the hand-rolled loss against `scipy`'s cross-entropy — the CLIP
   loss *is* two cross-entropies stacked back to back.
3. **Visualize** the similarity matrix and the training curve, with what to notice.
4. **Zero-shot** classification and prompt-template averaging.
5. **Tradeoffs & gotchas** — temperature, $\ell_2$-normalisation, batch size,
   and the modality gap — each with a runnable demo.
6. **Your turn** — implement the symmetric loss yourself.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

rng = np.random.default_rng(7)


## 1. Build two toy modalities with shared class structure

Real images and text live in totally different surface spaces, but a CLIP-style model has to recover the fact that an image of a cat and a sentence about cats share an underlying *class*. We simulate this by drawing all samples from a low-rank class-conditional structure in a hidden 4-D space, then projecting it into two **different** observation spaces: 64-D for "images" and 32-D for "text".


In [ ]:
N_CLASSES = 4
PER_CLASS = 16  # samples per class, per modality
HIDDEN = 4
D_IMG = 64
D_TXT = 32

# A class prototype in hidden space — the *shared* signal across modalities.
class_protos = rng.normal(size=(N_CLASSES, HIDDEN))

# Per-sample hidden code: class prototype + tiny noise.
def sample_hidden(n_samples_per_class, noise=0.3):
    hidden = []
    labels = []
    for c in range(N_CLASSES):
        block = class_protos[c] + noise * rng.normal(size=(n_samples_per_class, HIDDEN))
        hidden.append(block)
        labels.extend([c] * n_samples_per_class)
    return np.concatenate(hidden, axis=0), np.array(labels)

# Two *independent* random projections into the two surface spaces.
P_img = rng.normal(size=(HIDDEN, D_IMG)) / np.sqrt(HIDDEN)
P_txt = rng.normal(size=(HIDDEN, D_TXT)) / np.sqrt(HIDDEN)

# Build the matched pairs: the i-th image and i-th text share the same hidden code.
h_train, y_train = sample_hidden(PER_CLASS, noise=0.3)
image_features_train = h_train @ P_img + 0.4 * rng.normal(size=(len(h_train), D_IMG))
text_features_train  = h_train @ P_txt + 0.4 * rng.normal(size=(len(h_train), D_TXT))

print('image_features_train:', image_features_train.shape)
print('text_features_train :', text_features_train.shape)
print('labels              :', y_train[:10], '…')


## 2. Learnable encoders — one linear layer each into a shared $d$-dim space

This is the smallest possible CLIP. Each encoder is one matrix, the output is $\ell_2$-normalised, and the similarity is a dot product on unit vectors.


In [ ]:
D = 16  # shared embedding dim

# Init encoder matrices.
W_img = rng.normal(size=(D_IMG, D)) / np.sqrt(D_IMG)
W_txt = rng.normal(size=(D_TXT, D)) / np.sqrt(D_TXT)

def l2_normalise(X, eps=1e-9):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)

def encode_img(X, W=W_img):
    return l2_normalise(X @ W)

def encode_txt(X, W=W_txt):
    return l2_normalise(X @ W)

# Sanity check: every row has unit norm.
z_img = encode_img(image_features_train)
z_txt = encode_txt(text_features_train)
print('||z_img||:', np.linalg.norm(z_img, axis=1)[:5])
print('||z_txt||:', np.linalg.norm(z_txt, axis=1)[:5])


## 3. The symmetric CLIP loss + a hand-rolled gradient

Given a batch of $B$ matched pairs and a temperature $\tau$:

$$
S_{ij} = z^{\text{img}}_i \cdot z^{\text{txt}}_j / \tau
$$

$$
\mathcal{L} = \tfrac{1}{2}(\mathrm{CE}(\text{softmax}_\text{row}(S),\ I) + \mathrm{CE}(\text{softmax}_\text{col}(S),\ I))
$$

We compute the loss in NumPy and use a finite-difference numerical gradient — slow but transparent. Good enough to actually train the toy.


In [ ]:
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def clip_loss(z_img, z_txt, tau=0.1):
    """Symmetric CLIP loss on already-normalised embeddings."""
    B = z_img.shape[0]
    S = z_img @ z_txt.T / tau               # B x B logits
    p_rows = softmax(S, axis=1)             # P(text | image)
    p_cols = softmax(S, axis=0)             # P(image | text)
    diag_idx = np.arange(B)
    loss_i2t = -np.log(p_rows[diag_idx, diag_idx] + 1e-12).mean()
    loss_t2i = -np.log(p_cols[diag_idx, diag_idx] + 1e-12).mean()
    return 0.5 * (loss_i2t + loss_t2i)

# Quick sanity check: at random init the loss should be ~ log B.
B = image_features_train.shape[0]
init_loss = clip_loss(encode_img(image_features_train), encode_txt(text_features_train), tau=0.1)
print(f'B = {B}, log B = {np.log(B):.3f}, init loss = {init_loss:.3f}')


### Validate: the CLIP loss is just two cross-entropies

There is no `sklearn.CLIP`, but the loss is not exotic — each direction is an
ordinary softmax **cross-entropy** whose labels are the diagonal (`image i` ↔
`text i`). We confirm our hand-rolled `clip_loss` equals the mean of
`scipy.special`'s log-softmax evaluated at the true class, in both directions.

In [ ]:
from scipy.special import log_softmax as sp_log_softmax

def clip_loss_via_scipy(z_img, z_txt, tau=0.1):
    B = z_img.shape[0]
    S = z_img @ z_txt.T / tau
    diag = np.arange(B)
    # row direction: for each image i, cross-entropy of picking text i
    ce_rows = -sp_log_softmax(S, axis=1)[diag, diag].mean()
    # column direction: for each text j, cross-entropy of picking image j
    ce_cols = -sp_log_softmax(S, axis=0)[diag, diag].mean()
    return 0.5 * (ce_rows + ce_cols)

z_i0 = encode_img(image_features_train)
z_t0 = encode_txt(text_features_train)
ours   = clip_loss(z_i0, z_t0, tau=0.1)
theirs = clip_loss_via_scipy(z_i0, z_t0, tau=0.1)
print(f'from-scratch clip_loss     : {ours:.6f}')
print(f'scipy symmetric cross-ent  : {theirs:.6f}')
assert np.isclose(ours, theirs, atol=1e-9), 'CLIP loss must equal symmetric cross-entropy'
print('\n✅ the CLIP objective IS symmetric cross-entropy with diagonal labels')

### Validate: the CLIP loss is just two cross-entropies

There is no `sklearn.CLIP`, but the loss is not exotic — each direction is an
ordinary softmax **cross-entropy** whose labels are the diagonal (`image i` ↔
`text i`). We confirm our hand-rolled `clip_loss` equals the mean of
`scipy.special`'s log-softmax evaluated at the true class, in both directions.

In [ ]:
from scipy.special import log_softmax as sp_log_softmax

def clip_loss_via_scipy(z_img, z_txt, tau=0.1):
    B = z_img.shape[0]
    S = z_img @ z_txt.T / tau
    diag = np.arange(B)
    # row direction: for each image i, cross-entropy of picking text i
    ce_rows = -sp_log_softmax(S, axis=1)[diag, diag].mean()
    # column direction: for each text j, cross-entropy of picking image j
    ce_cols = -sp_log_softmax(S, axis=0)[diag, diag].mean()
    return 0.5 * (ce_rows + ce_cols)

z_i0 = encode_img(image_features_train)
z_t0 = encode_txt(text_features_train)
ours   = clip_loss(z_i0, z_t0, tau=0.1)
theirs = clip_loss_via_scipy(z_i0, z_t0, tau=0.1)
print(f'from-scratch clip_loss     : {ours:.6f}')
print(f'scipy symmetric cross-ent  : {theirs:.6f}')
assert np.isclose(ours, theirs, atol=1e-9), 'CLIP loss must equal symmetric cross-entropy'
print('\n✅ the CLIP objective IS symmetric cross-entropy with diagonal labels')

**What to notice — the training curve.** The loss starts near $\log B$ (chance:
the softmax is uniform, so the diagonal probability is $\approx 1/B$) and falls
as the two encoders rotate their outputs into agreement. It cannot go below 0,
and in practice it plateaus well above 0 because a single linear layer per
modality has limited capacity — real CLIP uses deep ViT/Transformer encoders.

## 4. Train with finite-difference gradient descent

For pedagogy we differentiate the loss with a tiny finite-difference probe rather than coding the analytic gradient by hand. This is slow but correct, and the toy is small enough that ~500 steps finish in a few seconds.


In [ ]:
def loss_for(W_img_, W_txt_, tau=0.1):
    z_i = l2_normalise(image_features_train @ W_img_)
    z_t = l2_normalise(text_features_train @ W_txt_)
    return clip_loss(z_i, z_t, tau=tau)

def numerical_grad(W, fn, eps=1e-3):
    g = np.zeros_like(W)
    base = fn(W)
    flat = W.reshape(-1)
    grad_flat = g.reshape(-1)
    # Probe in a small random subspace to keep this fast (proper SGD-of-gradient
    # would do the whole matrix; the random subset is cheaper and noisy but
    # works for the toy.)
    idx = rng.choice(flat.size, size=min(96, flat.size), replace=False)
    for k in idx:
        flat[k] += eps
        plus = fn(W)
        flat[k] -= 2 * eps
        minus = fn(W)
        flat[k] += eps  # restore
        grad_flat[k] = (plus - minus) / (2 * eps)
    return g

losses = []
lr = 0.6
TAU = 0.1
W_img_train = W_img.copy()
W_txt_train = W_txt.copy()
for step in range(500):
    g_img = numerical_grad(W_img_train, lambda W: loss_for(W, W_txt_train, tau=TAU))
    g_txt = numerical_grad(W_txt_train, lambda W: loss_for(W_img_train, W, tau=TAU))
    W_img_train -= lr * g_img
    W_txt_train -= lr * g_txt
    losses.append(loss_for(W_img_train, W_txt_train, tau=TAU))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, color='#6366f1', linewidth=2)
ax.axhline(np.log(B), color='#f97316', linestyle=':', alpha=0.7, label=f'log B = {np.log(B):.2f} (chance)')
ax.set_xlabel('step')
ax.set_ylabel('symmetric CLIP loss')
ax.set_title('Training the toy CLIP (NumPy)', fontsize=12)
ax.legend(facecolor='#1a1d27', edgecolor='#2a2d3a')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'final loss: {losses[-1]:.3f}  (vs random init {init_loss:.3f})')


**What to notice — the similarity matrix.** After training, the **diagonal**
(matched image↔text) is brighter than the off-diagonal (mismatched) entries.
That bright diagonal is *exactly and only* what the loss optimises: pull the
positive pair together, push every in-batch negative apart. Retrieval accuracy =
"is the brightest cell in each row on the diagonal?" 

**What to notice — the training curve.** The loss starts near $\log B$ (chance:
the softmax is uniform, so the diagonal probability is $\approx 1/B$) and falls
as the two encoders rotate their outputs into agreement. It cannot go below 0,
and in practice it plateaus well above 0 because a single linear layer per
modality has limited capacity — real CLIP uses deep ViT/Transformer encoders.

## 5. The similarity matrix on the trained batch — the diagonal should light up

The CLIP loss directly optimises the diagonal of $S$ on each training batch. We visualise that diagonal lighting up on a small slice of the training set. (Generalising to truly held-out *pairwise* retrieval would take orders of magnitude more data and capacity than a single linear layer — the toy here is just to make the training objective concrete. Section 6 shows that **class-level** generalisation already works with these toy encoders, which is the more useful demo.)

**What to notice — zero-shot works on held-out images.** These test images were
never seen in training, yet classifying them by nearest **text**-derived
prototype beats chance ($1/4 = 25\%$). That is the whole CLIP inference trick:
labels are just more text, so a new class costs one prompt, not one retraining.

In [ ]:
# Visualise a slice of the trained-batch similarity matrix.
B_show = 8
slice_idx = rng.choice(image_features_train.shape[0], size=B_show, replace=False)

z_img_slice = l2_normalise(image_features_train[slice_idx] @ W_img_train)
z_txt_slice = l2_normalise(text_features_train[slice_idx] @ W_txt_train)
S_slice = z_img_slice @ z_txt_slice.T

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(S_slice, cmap='viridis')
ax.set_xlabel('text index')
ax.set_ylabel('image index')
ax.set_title('Trained-batch cosine-similarity matrix\n(diagonal positives should dominate)', fontsize=11)
plt.colorbar(im, ax=ax, label='cos(z_img, z_txt)')
plt.tight_layout()
plt.show()

# In-batch retrieval accuracy — what the contrastive loss directly optimises.
preds = S_slice.argmax(axis=1)
acc = (preds == np.arange(B_show)).mean()
print(f'in-batch image -> text top-1 retrieval accuracy: {acc:.2%}')

**What to notice — the similarity matrix.** After training, the **diagonal**
(matched image↔text) is brighter than the off-diagonal (mismatched) entries.
That bright diagonal is *exactly and only* what the loss optimises: pull the
positive pair together, push every in-batch negative apart. Retrieval accuracy =
"is the brightest cell in each row on the diagonal?" 

**What to notice — prompt ensembling.** Averaging several noisy "template"
embeddings per class and re-normalising beats a single template, because the
template-specific noise is zero-mean and cancels while the shared class signal
survives. Real CLIP does exactly this with `"a photo of a {}"`, `"a blurry
photo of a {}"`, … — a free accuracy bump at inference time.

## Tradeoffs & gotchas

Four things that decide whether a contrastive model trains at all. Each has a
runnable demo below.

| Knob | Too small | Too large | Rule of thumb |
|------|-----------|-----------|---------------|
| temperature $\tau$ | logits explode, one negative dominates | softmax → uniform, no learning signal | $\tau \approx 0.07$ (learned in real CLIP) |
| $\ell_2$-normalise | — | — | **required**: puts all embeddings on a sphere so dot product = cosine |
| batch size $B$ | few negatives, weak contrast | more memory | bigger is better; CLIP used 32k |
| modality gap | — | — | image and text clouds stay in *separate cones*; retrieval still works |

### Gotcha 1 — temperature $\tau$ scales the contrast

$\tau$ divides the logits. Small $\tau$ sharpens the softmax (one negative can
dominate the gradient); large $\tau$ flattens it toward uniform, where the
diagonal probability is $\approx 1/B$ and the loss sits at its $\log B$ floor
with almost no gradient. We sweep $\tau$ on the *trained* embeddings.

In [ ]:
z_i = l2_normalise(image_features_train @ W_img_train)
z_t = l2_normalise(text_features_train @ W_txt_train)
print(f'{"tau":>6} | {"loss":>7} | {"log B floor":>11}')
for tau in [0.02, 0.1, 0.5, 2.0, 10.0]:
    L = clip_loss(z_i, z_t, tau=tau)
    print(f'{tau:6.2f} | {L:7.3f} | {np.log(len(z_i)):11.3f}')
print('\nAs tau grows the loss climbs toward the log B chance floor: the softmax flattens')
print('and the objective stops distinguishing the positive from the negatives.')

### Gotcha 2 — $\ell_2$-normalisation makes the score depend on *direction only*

Without normalisation the dot product mixes *direction* and *magnitude*, so the
encoder can lower the loss by simply inflating its output norm instead of
aligning directions — and the loss is no longer scale-invariant. Normalising
every embedding to the unit sphere makes the score a pure **cosine**, bounds
each logit to $[-1/\tau, +1/\tau]$, and makes the loss immune to a rescaling of
the encoder. We verify that invariance.

In [ ]:
raw_i = image_features_train @ W_img_train      # un-normalised
raw_t = text_features_train  @ W_txt_train

def _loss_no_norm(a, b, tau=0.1):
    B = a.shape[0]
    Sm = a @ b.T / tau
    d = np.arange(B)
    li = -np.log(softmax(Sm, 1)[d, d] + 1e-12).mean()
    lt = -np.log(softmax(Sm, 0)[d, d] + 1e-12).mean()
    return 0.5 * (li + lt)

# Scale the image encoder's output by 3x. Cosine (normalised) is unchanged;
# the raw dot-product loss changes because it saw bigger logits.
loss_norm_1x = clip_loss(l2_normalise(raw_i),     l2_normalise(raw_t))
loss_norm_3x = clip_loss(l2_normalise(3 * raw_i), l2_normalise(raw_t))
loss_raw_1x  = _loss_no_norm(raw_i,     raw_t)
loss_raw_3x  = _loss_no_norm(3 * raw_i, raw_t)
print(f'{"":>18} | {"1x":>8} | {"3x scale":>8}')
print(f'{"normalised (cos)":>18} | {loss_norm_1x:8.4f} | {loss_norm_3x:8.4f}')
print(f'{"raw dot product":>18} | {loss_raw_1x:8.4f} | {loss_raw_3x:8.4f}')
assert np.isclose(loss_norm_1x, loss_norm_3x), 'cosine loss must be scale-invariant'
assert not np.isclose(loss_raw_1x, loss_raw_3x), 'raw-dot loss changes with scale'
print('\nNormalised: identical (direction only). Raw: changes — the model could game the')
print('loss by growing norms instead of learning alignment. That is why CLIP normalises.')

### Gotcha 3 — the modality gap

A surprising empirical fact: even after training, image embeddings and text
embeddings occupy **separate cones** on the sphere — the average image
embedding and the average text embedding are far apart. Retrieval still works
because it only cares about *relative* distances within a modality's neighbours,
not about closing the absolute gap. We measure it here.

In [ ]:
img_mean = l2_normalise((z_i.mean(0))[None])[0]
txt_mean = l2_normalise((z_t.mean(0))[None])[0]
gap = np.linalg.norm(img_mean - txt_mean)
# average within-modality neighbour distance for comparison
within_img = np.mean([np.linalg.norm(z_i[a] - z_i[b])
                      for a in range(len(z_i)) for b in range(len(z_i)) if a != b])
print(f'distance between modality centroids : {gap:.3f}')
print(f'avg within-image pairwise distance  : {within_img:.3f}')
print('\nThe two modalities sit in different regions of the sphere (the modality gap),')
print('yet zero-shot classification above still beat chance — matching is relative, not absolute.')

## 6. Zero-shot classification via class prototypes

The whole point of CLIP at inference time: build a *text* embedding per class, then classify any new image by nearest text prototype. Here we average the trained text embeddings within each class to form prototypes.


In [ ]:
# Build a class prototype in shared space by averaging the trained text
# embeddings within each class on the training set.
z_txt_train = l2_normalise(text_features_train @ W_txt_train)
prototypes = np.zeros((N_CLASSES, D))
for c in range(N_CLASSES):
    mask = y_train == c
    proto = z_txt_train[mask].mean(axis=0)
    prototypes[c] = proto / (np.linalg.norm(proto) + 1e-9)

# Fresh held-out images we have *never* trained on.
h_test, y_test = sample_hidden(8, noise=0.3)
img_test = h_test @ P_img + 0.4 * rng.normal(size=(len(h_test), D_IMG))

# Encode held-out images, classify by nearest *text-derived* class prototype.
z_img_test_proto = l2_normalise(img_test @ W_img_train)
scores = z_img_test_proto @ prototypes.T   # (N_test, N_CLASSES)
preds = scores.argmax(axis=1)
acc = (preds == y_test).mean()
print(f'zero-shot classification accuracy on held-out images: {acc:.2%}')
print(f'(chance = {100 / N_CLASSES:.1f}%)')

**What to notice — zero-shot works on held-out images.** These test images were
never seen in training, yet classifying them by nearest **text**-derived
prototype beats chance ($1/4 = 25\%$). That is the whole CLIP inference trick:
labels are just more text, so a new class costs one prompt, not one retraining.

## 7. Averaging multiple "prompt templates" beats one template

We can't conjure real LLM tokens here, but we *can* simulate the effect of using multiple prompt templates by adding small independent noise to a prompt embedding and then averaging. This is structurally identical to averaging `"a photo of a cat"`, `"a picture of a cat"`, `"a close-up of a cat"`, ... at inference time in real CLIP.


## Key takeaways

- **CLIP learns a shared space, not a label set.** Two encoders map images and
  text into one $d$-dim sphere; matched pairs are pulled together, mismatched
  pairs pushed apart.
- **The loss is symmetric cross-entropy** with the batch diagonal as labels —
  we verified `clip_loss == scipy`'s two-direction cross-entropy exactly.
- **Zero-shot = nearest text prototype.** New classes cost one prompt, not one
  retraining; prompt-template averaging cancels noise for a free accuracy bump.
- **The knobs that matter:** temperature $\tau$ (sharpness of contrast),
  $\ell_2$-normalisation (makes the score a bounded cosine), and batch size
  (number of negatives). The **modality gap** persists but doesn't break retrieval.
- Our toy uses one linear layer per modality and finite-difference gradients;
  real CLIP swaps in deep ViT/Transformer encoders, analytic backprop, and
  32k-example batches — the objective is identical.